## Exploratory Testing 

In [1]:
# exploratory comparison - Cyclic ID vs Standard ID

# ==================================================
# Compare the cyclic ID (on a cyclic graph) vs standard ID (on an acyclic graph)
# - 20 single-gene perturbations
# - 20 multi-gene perturbations
# Total - 40 comparison runs
# ====================================================

# imports needed
import pandas as pd
import time
from y0.dsl import Variable, P
import networkx as nx
from y0.algorithm.identify import Unidentifiable, identify_outcomes
from y0.graph import NxMixedGraph
from y0.algorithm.identify.utils import Identification



In [2]:
## Load E. coli GRN 

ecoli_graph = nx.read_graphml("ecoli_full_network_no_small_rna.graphml")

print(f"✓ Network loaded (NetworkX):")
print(f"  - Nodes: {ecoli_graph.number_of_nodes():,}")
print(f"  - Edges: {ecoli_graph.number_of_edges():,}")
print(f"  - Is DAG: {nx.is_directed_acyclic_graph(ecoli_graph)}")

# Convert to NxMixedGraph for causal inference
ecoli_graph = NxMixedGraph.from_edges(
    directed=list(ecoli_graph.edges())
)

print(f"\n✓ Converted to NxMixedGraph")
print(f"  - Total nodes: {len(ecoli_graph):,}")
print(f"  - Directed edges: {ecoli_graph.directed.number_of_edges():,}")
print(f"  - Undirected edges: {ecoli_graph.undirected.number_of_edges():,}")



✓ Network loaded (NetworkX):
  - Nodes: 2,976
  - Edges: 9,211
  - Is DAG: False

✓ Converted to NxMixedGraph
  - Total nodes: 2,962
  - Directed edges: 9,211
  - Undirected edges: 0


In [3]:
# pre processing step 

start = time.time()
ecoli_admg = ecoli_graph.to_admg()
admg_conversion_time = time.time() - start

print(f"Converted to ADMG:")
print(f"  - Conversion time: {admg_conversion_time:.2f} seconds")
print(f"  - ADMG vertices: {len(ecoli_admg.vertices):,}")
print(f"  - Directed edges: {len(ecoli_admg.di_edges):,}")
print(f"  - Bidirected edges: {len(ecoli_admg.bi_edges):,}")
print(f"\n✓ ADMG ready for standard ID algorithm")

ModuleNotFoundError: No module named 'ananke'

In [12]:
# load the test cases from CSV

# single gene perturbations
df_single = pd.read_csv("ecoli_benchmark_all_combined.csv")
print(f"✓ Loaded single-gene data: {len(df_single):,} queries")
print(f"  - Identifiable: {df_single['identifiable'].sum():,}")
print(f"  - Unidentifiable: {(~df_single['identifiable']).sum():,}")

# Multi-gene perturbations
df_multi = pd.read_csv('ecoli_5x5_1000samples_FINAL.csv')
print(f"\n✓ Loaded multi-gene data: {len(df_multi):,} queries")
print(f"  - Identifiable: {df_multi['identifiable'].sum():,}")
print(f"  - Unidentifiable: {(~df_multi['identifiable']).sum():,}")

# Select test queries
print("\n" + "="*60)
print("SELECTING TEST QUERIES")
print("="*60)

# Single-gene: 10 identifiable + 10 unidentifiable
single_identifiable = df_single[df_single['identifiable'] == True].sample(n=10, random_state=42)
single_unidentifiable = df_single[df_single['identifiable'] == False].sample(n=10, random_state=42)

# Multi-gene: 10 identifiable + 10 unidentifiable
multi_identifiable = df_multi[df_multi['identifiable'] == True].sample(n=10, random_state=42)
multi_unidentifiable = df_multi[df_multi['identifiable'] == False].sample(n=10, random_state=42)

print(f"\n✓ Selected 40 test queries:")
print(f"  - Single-gene identifiable: {len(single_identifiable)}")
print(f"  - Single-gene unidentifiable: {len(single_unidentifiable)}")
print(f"  - Multi-gene identifiable: {len(multi_identifiable)}")
print(f"  - Multi-gene unidentifiable: {len(multi_unidentifiable)}")

✓ Loaded single-gene data: 555 queries
  - Identifiable: 63
  - Unidentifiable: 492

✓ Loaded multi-gene data: 1,000 queries
  - Identifiable: 893
  - Unidentifiable: 107

SELECTING TEST QUERIES

✓ Selected 40 test queries:
  - Single-gene identifiable: 10
  - Single-gene unidentifiable: 10
  - Multi-gene identifiable: 10
  - Multi-gene unidentifiable: 10


In [ ]:
# comparison function

def compare_cyclic_vs_standardID(
    admg,                    # pre processed ADMG
    intervention,            # Gene name(s) - string or list
    target,                  # Gene name(s) - string or list
    cyclic_result_from_csv   # Boolean from CSV
):
    
    results = {
        'intervention': str(intervention),
        'target': str(target),
        'cyclic_id_result': cyclic_result_from_csv  # Use CSV value
    }
    
    # run standard ID algorithm on the ADMG
    
    start = time.time()
    try:
        # Convert to Variable objects
        if isinstance(target, str):
            target_var = {Variable(target)}
        else:
            target_var = {Variable(g.strip()) for g in target}
            
        if isinstance(intervention, str):
            intervention_var = {Variable(intervention)}
        else:
            intervention_var = {Variable(g.strip()) for g in intervention}
        
        # Run standard ID on ADMG
        estimand = identify_outcomes(
            graph=admg,
            outcomes=target_var,
            treatments=intervention_var
        )
        results['standard_id_result'] = True
        
    except Unidentifiable:
        results['standard_id_result'] = False
    except Exception as e:
        results['standard_id_result'] = False
        results['standard_id_error'] = str(e)[:200]
        
    results['standard_id_time'] = time.time() - start
    
    # Compare results
    
    results['results_match'] = (
        results['cyclic_id_result'] == results['standard_id_result']
    )
    
    return results

print("✓ Comparison function defined")

✓ Comparison function defined


In [ ]:
# Comparison of results by calling function

comparison_results = [] # list to hold results

# single gene identifiable (10 queries)
print("\n[1/4] Running 10 single-gene identifiable queries.. ") 
for i, (idx, row) in enumerate(single_identifiable.iterrows(), 1):
    print(f"Query {i}/10: {row['intervention']} -> {row['target']}", end="...")
    
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_admg,
        intervention=row['intervention'],
        target=row['target'],
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'single_identifiable' # label added to tell which query is which
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" (std: {result['standard_id_time']:.2f}s)")


# -------------------------------------------------------------------

# single gene unidentifiable (10 queries)
print("\n[2/4] Running 10 single-gene unidentifiable queries.. ")
for i, (idx, row) in enumerate(single_unidentifiable.iterrows(), 1):
    print(f"Query {i}/10: {row['intervention']} -> {row['target']}", end="...")
    
    
    result = compare_cyclic_vs_standardID(
        admg=ecoli_admg,
        intervention=row['intervention'],
        target=row['target'],
        cyclic_result_from_csv=row['identifiable']
    )
    
    result['query_type'] = 'single_unidentifiable' 
    result['query_number'] = i
    comparison_results.append(result)
    
    print(f" (std: {result['standard_id_time']:.2f}s)")
    
# -------------------------------------------------------------------
# Multi gene 
# -------------------------------------------------------------------
